# Introduction to Python Generators

Python **generators** are a powerful tool for creating iterators using a simple, memory-efficient syntax. Unlike regular functions that return a single value and terminate, generators **yield** values one at a time and pause their execution until the next value is requested.

## 1. A Simple Generator Example

A generator function looks exactly like a normal function, but it uses the `yield` keyword instead of `return`. 

When a generator function is called, it does not run the code immediately. Instead, it returns a **generator object**. Each time you call `next()` on this object, the function runs until it hits a `yield` statement, gives back that value, and pauses.

In [ ]:
def count_up_to(max_value):
    """Yields numbers from 1 up to max_value."""
    count = 1
    while count <= max_value:
        yield count
        count += 1

# Create the generator object
counter = count_up_to(3)
print("Generator object created:", counter)

### Consuming the Generator

You can retrieve values one by one using the `next()` function, or loop through them cleanly using a `for` loop.

In [ ]:
# Fetching values manually
print(next(counter))  # Outputs: 1
print(next(counter))  # Outputs: 2
print(next(counter))  # Outputs: 3

# The next call would raise a StopIteration exception because the generator is exhausted.
# print(next(counter))

In [ ]:
# Using a for loop automatically handles the generator execution and stops when done
for number in count_up_to(3):
    print(f"Loop value: {number}")

## 2. Basic Prime Number Generator

Generators are ideal for handling **infinite streams of data** because they calculate values on-the-fly without storing the entire sequence in memory. Here is a naive approach checking each number manually.

In [ ]:
def is_prime(n):
    """Helper function to check if a number is prime."""
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

def prime_generator():
    """An infinite generator that yields sequential prime numbers."""
    number = 2
    while True:
        if is_prime(number):
            yield number
        number += 1

## 3. Optimizing with the Incremental Sieve of Eratosthenes

The standard **Sieve of Eratosthenes** requires a predefined upper bound because it works by marking composite numbers in a fixed array. 

To keep our generator **infinite and memory efficient**, we use an **Incremental Sieve**. Instead of a massive array, we use a dictionary to track composite numbers dynamically. The keys are the next composite numbers we expect, and the values are the prime factors that get us there.

In [ ]:
def sieve_prime_generator():
    """An infinite generator yielding primes using an incremental Sieve of Eratosthenes."""
    # Map: next_composite_number -> list of prime steps that divide it
    composites = {}
    
    number = 2
    while True:
        if number not in composites:
            # If the number is not in our dictionary, it is a Prime!
            yield number
            # Record its square as the next composite to watch out for
            composites[number * number] = [number]
        else:
            # If it is in the dictionary, it's a composite number.
            # We push the dividing prime factors forward to their next multiples.
            for prime in composites[number]:
                next_multiple = number + prime
                composites.setdefault(next_multiple, []).append(prime)
            
            # Delete the current number from memory to keep the dictionary small
            del composites[number]
            
        number += 1

### Comparing Performance

Let's test our new generator by grabbing the first 10 primes. Notice that the interface remains exactly the same as our previous generator (`next()`), but the internal mechanics are drastically faster for larger numbers.

In [ ]:
sieve_primes = sieve_prime_generator()

print("The first 10 prime numbers using the Sieve:")
for _ in range(10):
    print(next(sieve_primes))

### Summary of Benefits
* **Memory Efficiency**: Only one item is held in memory at a time. Creating a list of 1,000,000 primes takes massive memory; a generator takes virtually none.
* **Lazy Evaluation**: Calculations are only performed when you explicitly ask for the next result.
* **Algorithmic Boost**: The incremental sieve avoids division operations entirely, making it much more performant for generating high sequences of primes compared to trial division.